# 🍲 Synthetic Recipe Generation Pipeline

This script connects to your remote Open WebUI instance (running on the Mac mini) 
to automatically generate highly structured, budget-optimized recipes from a basic CSV catalogue.

In [24]:
import subprocess, sys

for pkg in ["pandas", "requests"]:
    try:
        __import__(pkg)
        print(f"✅ {pkg} already installed")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"✅ {pkg} installed")

✅ pandas already installed
✅ requests already installed


In [ ]:
import pandas as pd
import json
import time
import os
import requests

# --- CONFIGURATION ---
OPENWEBUI_BASE_URL = "https://chat.bebs.dev/api"
MODEL_NAME = "qwen3.5:9b-nothink"
OPEN_WEBUI_API_KEY = os.environ["OPENWEBUI_API_KEY"]  # set this in your shell, never hardcode

HEADERS = {
    "Authorization": f"Bearer {OPEN_WEBUI_API_KEY}",
    "Content-Type": "application/json",
}

INPUT_CSV = "../data/Nigerian Foods.csv"
OUTPUT_JSON = "../data/synthetic_nigerian_recipes.json"

print("✅ Configuration loaded. Using requests (same as curl).")

In [26]:
import requests

print("Testing connection to Open WebUI...")
try:
    r = requests.get(
        f"{OPENWEBUI_BASE_URL}/models",
        headers={"Authorization": f"Bearer {OPEN_WEBUI_API_KEY}"},
        timeout=15
    )
    print(f"Status: {r.status_code}")
    models = r.json().get("data", [])
    print(f"Available models: {[m['id'] for m in models]}")
except Exception as e:
    print(f"Connection failed: {e}")

Testing connection to Open WebUI...
Status: 200
Available models: ['qwen3.5:9b-nothink', 'qwen3:8b']
Status: 200
Available models: ['qwen3.5:9b-nothink', 'qwen3:8b']


In [27]:
def generate_recipe(food_name, description):
    """Prompts the remote LLM to generate a recipe using streaming to avoid Cloudflare 524 timeout."""

    prompt = f"""/no_think You are an expert chef specializing in budget-friendly African cuisine.
Create a detailed, authentic recipe for the following Nigerian dish: {food_name}.
Context about the dish: {description}

CRITICAL: Output ONLY valid JSON with NO markdown, NO extra text, NO explanations.
Do NOT use double quotes inside string values - use single quotes or escape them properly.
Do NOT include ANY text outside the JSON object.

Output this exact structure:
{{
  "title": "{food_name}",
  "cuisine_profile": "Nigerian",
  "nutrition": {{
    "calories_per_serving": 0,
    "servings": 4
  }},
  "ingredients_used": {{
    "anchor_ingredients": [{{"item": "Ingredient Name", "quantity": "amount"}}],
    "pantry_staples": [{{"item": "Spice/Oil Name", "quantity": "amount"}}]
  }},
  "instructions": [
    "Step 1...",
    "Step 2...",
    "Step 3...",
    "Step 4..."
  ]
}}"""

    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True  # streaming keeps Cloudflare from 524-ing the connection
    }

    try:
        r = requests.post(
            f"{OPENWEBUI_BASE_URL}/chat/completions",
            headers=HEADERS,
            json=payload,
            stream=True,
            timeout=300
        )
        r.raise_for_status()

        # Collect streamed SSE chunks into the full response
        full_content = ""
        for line in r.iter_lines():
            if not line:
                continue
            line = line.decode("utf-8")
            if not line.startswith("data: "):
                continue
            data = line[6:]
            if data == "[DONE]":
                break
            chunk = json.loads(data)
            delta = chunk["choices"][0]["delta"].get("content", "")
            if delta:
                full_content += delta

        # Strip markdown code fences if the model wraps JSON in them
        content = full_content.strip()
        if content.startswith("```"):
            content = content.split("```")[1]
            if content.startswith("json"):
                content = content[4:]

        content = content.strip()
        
        # Truncate at first closing brace if there's extra text after
        if content.count('}') > 0:
            last_brace = content.rfind('}')
            if last_brace > 0:
                content = content[:last_brace + 1]
        
        # DEBUG: Print the raw response before JSON parsing
        print(f"\n  [DEBUG] Raw response for {food_name} (length: {len(content)}):")
        print(f"  {content[:300]}...")  # Print first 300 chars for preview
        
        # Try to parse JSON, with error recovery
        try:
            return json.loads(content)
        except json.JSONDecodeError as e:
            print(f"  [JSON ERROR] {e}")
            print(f"  Attempting to fix JSON...")
            
            # Try to find and fix common issues
            # Fix double quotes at end of strings
            fixed_content = content.replace('."".', '."')
            fixed_content = fixed_content.replace('."",', '.",')
            fixed_content = fixed_content.replace('."" ]', '."]')
            
            # Try again
            try:
                return json.loads(fixed_content)
            except json.JSONDecodeError as e2:
                print(f"  [JSON ERROR AFTER FIX] {e2}")
                print(f"  Could not parse JSON for {food_name}")
                return None

    except Exception as e:
        print(f"  [ERROR] Failed to generate recipe for {food_name}: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"  [RESPONSE BODY] {e.response.text[:300]}")
        return None

print("✅ Generation function initialized.")

✅ Generation function initialized.


In [28]:
print(f"Loading {INPUT_CSV}...")
try:
    df = pd.read_csv(INPUT_CSV)
except FileNotFoundError:
    print(f"[!] Could not find {INPUT_CSV}. Make sure your Kaggle dataset is in the correct folder.")
    df = pd.DataFrame() # Create empty dataframe to prevent crash

if not df.empty:
    print(f"✅ Loaded {len(df)} items from the CSV.")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(df.head())
else:
    print("[!] CSV is empty or not loaded.")

Loading ../data/Nigerian Foods.csv...
✅ Loaded 109 items from the CSV.

Columns: ['Food_Name', 'Main_Ingredients', 'Description', 'Food_Health', 'Food_Class', 'Region', 'Spice_Level', 'Price_Range']

First few rows:
                   Food_Name  \
0                     Abacha   
1  Abacha and Ugba (Regular)   
2            Abacha and Ugba   
3         Afang Soup (Spicy)   
4                 Afang Soup   

                                    Main_Ingredients  \
0             African salad, utazi leaves, oil, fish   
1        Abacha (Cassava Fufu), Ugba (Oilbean Seeds)   
2  African salad, oil, fish, fermented oil bean s...   
3                   Afang Leaves, Water yam, Seafood   
4                Afang leaves, waterleaf, meat, fish   

                                         Description Food_Health   Food_Class  \
0                Cassava-based salad with spicy fish     Healthy  Traditional   
1   Shredded cassava fufu served with oilbean seeds.     Healthy  Traditional   
2  Cassava-

In [29]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_row(args):
    index, row, total = args
    food_name = row.get("Food_Name", f"Item {index}")
    description = row.get("Description", "Traditional Nigerian dish.")
    print(f"  [{index + 1}/{total}] Starting: {food_name}...")
    recipe = generate_recipe(food_name, description)
    if recipe:
        print(f"  [{index + 1}/{total}] ✅ Done: {food_name}")
    return recipe

if not df.empty:
    # Adjust WORKERS based on your Mac mini's RAM:
    # 9B model ≈ 6GB — if you have 16GB RAM, 2 workers is safe; 32GB → 4 workers
    WORKERS = 3

    test_df = df.head(5)   # remove .head(5) to run the full 109
    tasks = [(i, row, len(test_df)) for i, row in test_df.iterrows()]

    print(f"Generating {len(tasks)} recipes with {WORKERS} parallel workers...\n")
    synthetic_recipes = []

    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = {pool.submit(process_row, t): t for t in tasks}
        for future in as_completed(futures):
            result = future.result()
            if result:
                synthetic_recipes.append(result)

    print(f"\n✅ Successfully generated {len(synthetic_recipes)} recipes.")
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(synthetic_recipes, f, indent=2, ensure_ascii=False)
    print(f"📁 Saved to {OUTPUT_JSON}")
else:
    print("[!] Cannot generate recipes - CSV not loaded.")
    synthetic_recipes = []

Generating 5 recipes with 3 parallel workers...

  [1/5] Starting: Abacha...
  [2/5] Starting: Abacha and Ugba (Regular)...
  [3/5] Starting: Abacha and Ugba...

  [DEBUG] Raw response for Abacha and Ugba (Regular) (length: 1203):
  {
  "title": "Abacha and Ugba (Regular)",
  "cuisine_profile": "Nigerian",
  "nutrition": {
    "calories_per_serving": 350,
    "servings": 4
  },
  "ingredients_used": {
    "anchor_ingredients": [
      {"item": "Shredded Cassava (Abacha)", "quantity": "2 cups"},
      {"item": "Dried Oil Beans",...
  [2/5] ✅ Done: Abacha and Ugba (Regular)
  [4/5] Starting: Afang Soup (Spicy)...

  [DEBUG] Raw response for Abacha and Ugba (Regular) (length: 1203):
  {
  "title": "Abacha and Ugba (Regular)",
  "cuisine_profile": "Nigerian",
  "nutrition": {
    "calories_per_serving": 350,
    "servings": 4
  },
  "ingredients_used": {
    "anchor_ingredients": [
      {"item": "Shredded Cassava (Abacha)", "quantity": "2 cups"},
      {"item": "Dried Oil Beans",...
  [2/

In [30]:
# Preview the generated recipes
if synthetic_recipes:
    print("\n--- 📋 Output Preview (First Recipe) ---")
    print(json.dumps(synthetic_recipes[0], indent=2))
    print(f"\n--- 📊 Summary ---")
    print(f"Total recipes generated: {len(synthetic_recipes)}")
else:
    print("[!] No recipes were generated.")


--- 📋 Output Preview (First Recipe) ---
{
  "title": "Abacha and Ugba (Regular)",
  "cuisine_profile": "Nigerian",
  "nutrition": {
    "calories_per_serving": 350,
    "servings": 4
  },
  "ingredients_used": {
    "anchor_ingredients": [
      {
        "item": "Shredded Cassava (Abacha)",
        "quantity": "2 cups"
      },
      {
        "item": "Dried Oil Beans",
        "quantity": "1 cup soaked and cooked until tender"
      }
    ],
    "pantry_staples": [
      {
        "item": "Palm Oil",
        "quantity": "4 tablespoons"
      },
      {
        "item": "Scotch Bonnet Pepper",
        "quantity": "2-3 seeds minced"
      },
      {
        "item": "Onions (diced)",
        "quantity": "1 medium"
      },
      {
        "item": "Garlic",
        "quantity": "3 cloves"
      },
      {
        "item": "Salt and Stock Cube",
        "quantity": "to taste"
      }
    ]
  },
  "instructions": [
    "Wash the dried oil beans thoroughly, soak them overnight if using old st